##### Setup
- Explore which factors (columns) are associated with `cancel_yn` (churn), using the full cleaned dataset (no train/test split needed for this exploratory step)

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)

CLEAN = "read_parquet('df_clean_part*.parquet')"

##### Baseline Churn Rate
- Reference point for comparing segment-level churn rates below

In [ ]:
baseline = duckdb.sql(f"""
    SELECT round(count(*) FILTER (WHERE cancel_yn = '해지') * 100.0 / count(*), 4) AS baseline_churn_rate_pct
    FROM {CLEAN}
""").df()
baseline

,baseline_churn_rate_pct
0,4.0343


##### Categorical Columns: Churn Rate by Category
- For every categorical column, compute the churn rate (%) within each category value
- `deviation_from_baseline` shows how far each segment's churn rate is from the overall average — sorted by absolute deviation, so the strongest factors surface first
- Filters out tiny groups (`n < 1000`) to avoid noisy small-sample rates

In [7]:
categorical_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "AGMT_END_SEG", "BUNDLE_YN", "DIGITAL_GIGA_YN", "DIGITAL_ALOG_YN", "CH_LAST_DAYS_BF_GRP",
    "VOC_TOTAL_MONTH1_YN", "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM",
    "SMS_SEND_CLS_NM", "NFX_USE_YN", "YTB_USE_YN", "CH_FAV_RNK1",
]
# SQL에 들어가기 위해 여러 컬럼을 조회할 때 대괄호 없이 쉼표로 연결된 일반 텍스트
cols_sql = ", ".join(categorical_cols)
baseline_rate = baseline['baseline_churn_rate_pct'][0]

categorical_churn_scan = duckdb.sql(f"""
    WITH base AS (
        SELECT
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_cancelled,
            {cols_sql}
        FROM {CLEAN}
    ),
    unpvt AS (
        UNPIVOT base
        ON {cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        count(*) AS n,
        round(avg(is_cancelled) * 100, 4) AS churn_rate_pct,
        round(avg(is_cancelled) * 100 - {baseline_rate}, 4) AS deviation_from_baseline,
        round(avg(is_cancelled) * 100 / {baseline_rate}, 2) AS ratio_to_baseline
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(*) >= 1000
    ORDER BY abs(round(avg(is_cancelled) * 100 - {baseline_rate}, 4)) DESC
""").df()

categorical_churn_scan.head(30)

,column_name,value,n,churn_rate_pct,deviation_from_baseline,ratio_to_baseline
0,AGMT_END_SEG,약정만료전 1개월,181220,17.6548,13.6205,4.38
1,AGMT_END_SEG,약정만료전 1~2개월,205110,14.8886,10.8543,3.69
2,VOC_STOP_CANCEL_MONTH1_YN,Y,464918,13.8674,9.8331,3.44
3,AGMT_END_SEG,약정만료전 2~3개월,224681,12.0250,7.9907,2.98
4,AGE_GRP10,20대,185727,11.7560,7.7217,2.91
5,AGMT_END_SEG,약정만료후 1개월~2개월,128414,9.4686,5.4343,2.35
6,AGMT_END_SEG,약정만료 1개월,328269,8.9935,4.9592,2.23
7,PROD_NM_GRP,세이버,1652,8.4746,4.4403,2.10
8,AGMT_END_SEG,약정만료후 2개월~3개월,118056,8.3232,4.2889,2.06
9,AGMT_END_SEG,약정만료전 3~6개월,795058,8.2893,4.2550,2.05


##### Finding

`AGMT_END_SEG` shows up 8 times in the top 30, so we should check for multicollinearity before treating these as 8 separate signals.

A disproportionate share of cancellations come from customers in their 20s — this age group could be a good target for retention marketing.

`VOC_STOP_CANCEL_MONTH1_YN` isn't very useful on its own, since it only flags customers who've already decided to cancel. It'd be more valuable to look at why some customers who call about cancelling end up *not* cancelling.

##### Numeric Columns: Average Value by cancel_yn
- Compare the average of each numeric column between the `해지`(cancelled) and `유지`(retained) groups
- A large gap between the two rows suggests that column is informative for churn prediction

In [ ]:
numeric_cols = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM",
    "KIDS_USE_PV_MONTH1", "INHOME_RATE",
]

parts = []
for c in numeric_cols:
    expr = "round(avg(" + c + "), 3) AS avg_" + c
    parts.append(expr)

avg_exprs = ", ".join(parts)


numeric_churn_scan = duckdb.sql(f"""
    SELECT cancel_yn, count(*) AS n, {avg_exprs}
    FROM {CLEAN}
    GROUP BY cancel_yn
""").df()

numeric_churn_scan.T

,0,1
cancel_yn,해지,유지
n,889705,21163859
avg_TOTAL_USED_DAYS,2300.376,2777.906
avg_TV_SCRB,1.716,1.782
avg_ANALOG_SCRB,0.016,0.021
avg_DIGITAL_SCRB,1.7,1.761
avg_TOTAL_INTERNET_SCRB,0.656,0.489
avg_GIGA_INTERNET_SCRB,0.249,0.129
avg_TV_I_CNT,2.371,2.271
avg_CH_HH_AVG_MONTH1,3.218,4.613


##### Findings


##### Do Customers Still Appear After Their First `해지` Month?
- For each customer's first `해지` month, check whether they have any later row (`p_mt` after that month)
- If almost nobody appears afterward, that confirms `해지` = "left the dataset," so each churning customer contributes only one `해지` row out of their whole history

In [8]:
after_cancel_check = duckdb.sql(f"""
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    joined AS (
        SELECT
            fc.sha2_hash,
            fc.first_cancel_pmt,
            count(*) FILTER (WHERE c.p_mt > fc.first_cancel_pmt) AS n_rows_after
        FROM first_cancel fc
        JOIN {CLEAN} c ON c.sha2_hash = fc.sha2_hash
        GROUP BY fc.sha2_hash, fc.first_cancel_pmt
    )
    SELECT
        count(*) AS n_customers_with_cancel,
        count(*) FILTER (WHERE n_rows_after = 0) AS n_no_rows_after,
        count(*) FILTER (WHERE n_rows_after > 0) AS n_has_rows_after,
        round(count(*) FILTER (WHERE n_rows_after = 0) * 100.0 / count(*), 2) AS pct_no_rows_after
    FROM joined
""").df()

after_cancel_check

,n_customers_with_cancel,n_no_rows_after,n_has_rows_after,pct_no_rows_after
0,150613,15900,134713,10.56


##### What Does cancel_yn Look Like in the Rows After First `해지`?
- For the 134,713 customers who have rows after their first `해지`, check whether those later rows are mostly `유지` (reverted / reactivated) or mostly `해지` (status persists)

In [ ]:
rows_after_status = duckdb.sql(f"""
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    rows_after AS (
        SELECT c.cancel_yn
        FROM {CLEAN} c
        JOIN first_cancel fc ON c.sha2_hash = fc.sha2_hash
        WHERE c.p_mt > fc.first_cancel_pmt
    )
    SELECT
        cancel_yn,
        count(*) AS n,
        round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS pct
    FROM rows_after
    GROUP BY cancel_yn
""").df()

rows_after_status

,cancel_yn,n,pct
0,해지,739092,100.0


##### Do the Other Columns Also Freeze After First `해지`?
- We already know `cancel_yn` stays `해지` in every row after the first cancellation. Do the other columns (`TV_I_CNT`, `CH_HH_AVG_MONTH1`, etc.) also stay frozen at the same value, or do they keep updating month to month?
- For customers with 2+ rows after their first `해지`, check how many distinct values each column takes across those rows — `n_distinct=1` means frozen

In [10]:
other_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "INHOME_RATE", "AGMT_END_SEG", "AGMT_END_YMD", "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB",
    "DIGITAL_SCRB", "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "BUNDLE_YN", "DIGITAL_GIGA_YN",
    "DIGITAL_ALOG_YN", "TV_I_CNT", "CH_LAST_DAYS_BF_GRP", "VOC_TOTAL_MONTH1_YN",
    "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM", "SMS_SEND_CLS_NM",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM", "CH_FAV_RNK1",
    "KIDS_USE_PV_MONTH1", "NFX_USE_YN", "YTB_USE_YN",
]
# INHOME_RATE/CH_25_RATIO_MONTH1/etc. are DOUBLE while the rest are VARCHAR in the
# cleaned data — UNPIVOT needs matching types, so cast everything to VARCHAR first
other_cols_sql = ", ".join(other_cols)
cast_cols_sql = ", ".join(f"CAST({c} AS VARCHAR) AS {c}" for c in other_cols)

frozen_check = duckdb.sql(f"""
    WITH raw AS (
        SELECT sha2_hash, p_mt, cancel_yn, {cast_cols_sql}
        FROM {CLEAN}
    ),
    first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM raw
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    ),
    rows_after AS (
        SELECT r.*
        FROM raw r
        JOIN first_cancel fc ON r.sha2_hash = fc.sha2_hash
        WHERE r.p_mt > fc.first_cancel_pmt
    ),
    customers_with_2plus AS (
        SELECT sha2_hash
        FROM rows_after
        GROUP BY sha2_hash
        HAVING count(*) >= 2
    ),
    unpvt AS (
        UNPIVOT (
            SELECT sha2_hash, {other_cols_sql}
            FROM rows_after
            WHERE sha2_hash IN (SELECT sha2_hash FROM customers_with_2plus)
        )
        ON {other_cols_sql}
        INTO
            NAME column_name
            VALUE value
    ),
    per_customer_col AS (
        SELECT sha2_hash, column_name, count(DISTINCT value) AS n_distinct
        FROM unpvt
        GROUP BY sha2_hash, column_name
    )
    SELECT
        column_name,
        count(*) AS n_customers,
        round(avg(CASE WHEN n_distinct = 1 THEN 100.0 ELSE 0 END), 2) AS pct_frozen
    FROM per_customer_col
    GROUP BY column_name
    ORDER BY pct_frozen
""").df()

frozen_check

,column_name,n_customers,pct_frozen
0,TOTAL_USED_DAYS,120987,0.00
1,CH_HH_AVG_MONTH1,120987,23.96
2,CH_25_RATIO_MONTH1,120987,29.74
3,CH_25_RATIO_MEAN_3MM,120987,29.74
4,VOC_TOTAL_MONTH1_YN,120987,33.62
5,INHOME_RATE,95899,41.10
6,CH_FAV_RNK1,120987,49.16
7,AGMT_END_SEG,120987,59.83
8,CH_LAST_DAYS_BF_GRP,120987,60.59
9,VOC_STOP_CANCEL_MONTH1_YN,120987,64.33


---

## Part 2. Re-analysis After Removing Rows After the First Cancellation

##### Why Truncate
- Cancelled customers stay in the data as `해지` every month after cancelling (739,092 rows, `538e72fa`), so the same cancellation is counted many times
- Keep each customer's rows only up to their first `해지` month, as the view `churn_base` (no new file is written)
- From here on, churn rate means **monthly churn rate**: the chance that a not-yet-cancelled customer cancels in that month

In [5]:
import duckdb
import pandas as pd

# Self-contained: Part 2 can run without Part 1
pd.set_option('display.max_rows', None)
CLEAN = "read_parquet('df_clean_part*.parquet')"

duckdb.sql(f"""
    CREATE OR REPLACE VIEW churn_base AS
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    )
    SELECT c.*
    FROM {CLEAN} c
    LEFT JOIN first_cancel fc USING (sha2_hash)
    WHERE fc.first_cancel_pmt IS NULL          -- never cancelled: keep every month
       OR c.p_mt <= fc.first_cancel_pmt        -- cancelled: keep up to the first 해지 month
""")

# Sanity check: 22,053,564 - 739,092 = 21,314,472 rows; one 해지 row per churner (150,613)
truncate_check = duckdb.sql("""
    SELECT
        count(*) AS n_rows,
        count(*) FILTER (WHERE cancel_yn = '해지') AS n_cancel_rows,
        count(DISTINCT sha2_hash) AS n_customers,
        count(DISTINCT sha2_hash) FILTER (WHERE cancel_yn = '해지') AS n_churned_customers
    FROM churn_base
""").df()

truncate_check

,n_rows,n_cancel_rows,n_customers,n_churned_customers
0,21314472,150613,2135261,150613


In [6]:
baseline_trunc = duckdb.sql("""
    SELECT
        round(count(*) FILTER (WHERE cancel_yn = '해지') * 100.0 / count(*), 4) AS monthly_churn_rate_pct,
        round(count(DISTINCT sha2_hash) FILTER (WHERE cancel_yn = '해지') * 100.0
              / count(DISTINCT sha2_hash), 2) AS customer_churn_rate_pct
    FROM churn_base
""").df()

baseline_trunc

,monthly_churn_rate_pct,customer_churn_rate_pct
0,0.7066,7.05


In [7]:
categorical_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "AGMT_END_SEG", "BUNDLE_YN", "DIGITAL_GIGA_YN", "DIGITAL_ALOG_YN", "CH_LAST_DAYS_BF_GRP",
    "VOC_TOTAL_MONTH1_YN", "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10", "EMAIL_RECV_CLS_NM",
    "SMS_SEND_CLS_NM", "NFX_USE_YN", "YTB_USE_YN", "CH_FAV_RNK1",
]
cols_sql = ", ".join(categorical_cols)

baseline_rate_trunc = baseline_trunc['monthly_churn_rate_pct'][0]

categorical_churn_scan_trunc = duckdb.sql(f"""
    WITH base AS (
        SELECT
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_cancelled,
            {cols_sql}
        FROM churn_base
    ),
    unpvt AS (
        UNPIVOT base
        ON {cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        count(*) AS n,
        round(avg(is_cancelled) * 100, 4) AS churn_rate_pct,
        round(avg(is_cancelled) * 100 - {baseline_rate_trunc}, 4) AS deviation_from_baseline,
        round(avg(is_cancelled) * 100 / {baseline_rate_trunc}, 2) AS ratio_to_baseline
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(*) >= 1000
    ORDER BY ratio_to_baseline DESC
""").df()

categorical_churn_scan_trunc.head(30)

,column_name,value,n,churn_rate_pct,deviation_from_baseline,ratio_to_baseline
0,AGMT_END_SEG,약정만료전 1개월,152700,2.2750,1.5684,3.22
1,AGE_GRP10,20대,167704,2.2725,1.5659,3.22
2,AGMT_END_SEG,약정만료전 1~2개월,178577,2.2427,1.5361,3.17
3,VOC_STOP_CANCEL_MONTH1_YN,Y,408760,2.0340,1.3274,2.88
4,AGMT_END_SEG,약정만료전 2~3개월,201385,1.8482,1.1416,2.62
5,AGMT_END_SEG,약정만료전 3~6개월,740597,1.5452,0.8386,2.19
6,PAID_CHNL_CNT_GRP,3건 이상,20759,1.5415,0.8349,2.18
7,SVC_USE_DAYS_GRP,24개월~ 36개월미만,2091130,1.5266,0.8200,2.16
8,STB_RES_1M_YN,Y,2363358,1.5014,0.7948,2.12
9,PAID_CHNL_CNT_GRP,2건,68482,1.4150,0.7084,2.00


##### Before vs After Truncation
- The baseline changes (4.03% → 0.71%), so compare **ratios to baseline**, not raw rates

In [8]:
# "Before" = same scan on all cleaned rows (Part 1 method), recomputed here so this
# cell doesn't depend on Part 1 having been run
churn_scan_before = duckdb.sql(f"""
    WITH base AS (
        SELECT
            CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END AS is_cancelled,
            {cols_sql}
        FROM {CLEAN}
    ),
    unpvt AS (
        UNPIVOT base
        ON {cols_sql}
        INTO
            NAME column_name
            VALUE value
    )
    SELECT
        column_name,
        value,
        round(avg(is_cancelled) * 100, 4) AS rate_before,
        round(avg(is_cancelled) * 100
              / (SELECT avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END) * 100 FROM {CLEAN}), 2) AS ratio_before
    FROM unpvt
    GROUP BY column_name, value
    HAVING count(*) >= 1000
""").df()

compare_trunc = (
    churn_scan_before
    .merge(
        categorical_churn_scan_trunc[['column_name', 'value', 'n', 'churn_rate_pct', 'ratio_to_baseline']]
        .rename(columns={'churn_rate_pct': 'rate_after', 'ratio_to_baseline': 'ratio_after'}),
        on=['column_name', 'value'],
    )
    .sort_values('ratio_after', ascending=False)
)

compare_trunc.head(30)

,column_name,value,rate_before,ratio_before,n,rate_after,ratio_after
59,AGE_GRP10,20대,11.7560,2.91,167704,2.2725,3.22
60,AGMT_END_SEG,약정만료전 1개월,17.6548,4.38,152700,2.2750,3.22
58,AGMT_END_SEG,약정만료전 1~2개월,14.8886,3.69,178577,2.2427,3.17
12,VOC_STOP_CANCEL_MONTH1_YN,Y,13.8674,3.44,408760,2.0340,2.88
21,AGMT_END_SEG,약정만료전 2~3개월,12.0250,2.98,201385,1.8482,2.62
104,AGMT_END_SEG,약정만료전 3~6개월,8.2893,2.05,740597,1.5452,2.19
2,PAID_CHNL_CNT_GRP,3건 이상,7.7121,1.91,20759,1.5415,2.18
49,SVC_USE_DAYS_GRP,24개월~ 36개월미만,7.8075,1.94,2091130,1.5266,2.16
68,STB_RES_1M_YN,Y,8.2865,2.05,2363358,1.5014,2.12
23,PAID_CHNL_CNT_GRP,2건,7.1706,1.78,68482,1.4150,2.00


In [9]:
numeric_cols = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM",
    "KIDS_USE_PV_MONTH1", "INHOME_RATE",
]
avg_exprs = ", ".join(f"round(avg({c}), 3) AS avg_{c}" for c in numeric_cols)

numeric_churn_scan_trunc = duckdb.sql(f"""
    SELECT cancel_yn, count(*) AS n, {avg_exprs}
    FROM churn_base
    GROUP BY cancel_yn
""").df()

numeric_churn_scan_trunc.T

,0,1
cancel_yn,해지,유지
n,150613,21163859
avg_TOTAL_USED_DAYS,2183.926,2777.906
avg_TV_SCRB,1.713,1.782
avg_ANALOG_SCRB,0.016,0.021
avg_DIGITAL_SCRB,1.697,1.761
avg_TOTAL_INTERNET_SCRB,0.653,0.489
avg_GIGA_INTERNET_SCRB,0.246,0.129
avg_TV_I_CNT,2.366,2.271
avg_CH_HH_AVG_MONTH1,3.563,4.613


##### Findings
- 21,314,472 rows, 150,613 churned customers (one `해지` row each). Monthly churn rate **0.71%**
- Overstated risk corrected: 1 month before contract expiry 4.38x → **3.22x**
- Hidden risk revealed: tenure under 6 months 1.10x → **1.72x**
- Top 5 (vs baseline): 1 month before contract expiry 3.22x, age 20s 3.22x, 3+ paid channels 2.18x, no viewing in 3 months 1.83x, tenure under 6 months 1.72x
- Excluded: cancellation/suspension call (2.88x) looks like a signal of intent rather than a cause; tenure 24–36 months (2.16x) overlaps with contract expiry (Part 4)

---

## Part 3. Where Does Churn Come From: Churn Share by Contract Stage

##### Churn Share by Contract-Expiry Segment
- A high churn rate in a small segment still means few cancellations → look at **share of customers vs share of cancellations** together

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
CLEAN = "read_parquet('df_clean_part*.parquet')"

# Same truncated view as Part 2 (truncate-01) — recreated so this part is self-contained
duckdb.sql(f"""
    CREATE OR REPLACE VIEW churn_base AS
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    )
    SELECT c.*
    FROM {CLEAN} c
    LEFT JOIN first_cancel fc USING (sha2_hash)
    WHERE fc.first_cancel_pmt IS NULL
       OR c.p_mt <= fc.first_cancel_pmt
""")

n_months = duckdb.sql("SELECT count(DISTINCT p_mt) FROM churn_base").fetchone()[0]

# Chronological order: far before expiry -> expiry month -> long after expiry
EXPIRY_ORDER = [
    "약정만료전 12개월이상", "약정만료전 9~12개월", "약정만료전 6~9개월", "약정만료전 3~6개월",
    "약정만료전 2~3개월", "약정만료전 1~2개월", "약정만료전 1개월", "약정만료 1개월",
    "약정만료후 1개월~2개월", "약정만료후 2개월~3개월", "약정만료후 3~6개월",
    "약정만료후 6~9개월", "약정만료후 9~12개월", "약정만료후 12개월이상",
]

expiry_share = duckdb.sql(f"""
    WITH seg AS (
        SELECT
            AGMT_END_SEG,
            count(*) AS n,
            count(*) FILTER (WHERE cancel_yn = '해지') AS n_churn
        FROM churn_base
        GROUP BY AGMT_END_SEG
        HAVING count(*) >= 1000                 -- drops '기타' (33 rows)
    ),
    tot AS (
        SELECT sum(n) AS total_n, sum(n_churn) AS total_churn FROM seg
    )
    SELECT
        AGMT_END_SEG,
        n,
        n_churn,
        round(n_churn * 100.0 / n, 4) AS churn_rate_pct,
        round(n * 100.0 / total_n, 2) AS base_share_pct,
        round(n_churn * 100.0 / total_churn, 2) AS churn_share_pct,
        round((n_churn * 1.0 / total_churn) / (n * 1.0 / total_n), 2) AS concentration,
        round(n * 1.0 / {n_months}) AS avg_customers_per_month
    FROM seg, tot
""").df()

expiry_share['AGMT_END_SEG'] = pd.Categorical(expiry_share['AGMT_END_SEG'], EXPIRY_ORDER, ordered=True)
expiry_share = expiry_share.sort_values('AGMT_END_SEG').reset_index(drop=True)

print(f"months in data: {n_months}")
expiry_share

months in data: 11


,AGMT_END_SEG,n,n_churn,churn_rate_pct,base_share_pct,churn_share_pct,concentration,avg_customers_per_month
0,약정만료전 12개월이상,9198698,42187,0.4586,43.16,28.01,0.65,836245.0
1,약정만료전 9~12개월,1060600,6864,0.6472,4.98,4.56,0.92,96418.0
2,약정만료전 6~9개월,926090,9971,1.0767,4.34,6.62,1.52,84190.0
3,약정만료전 3~6개월,740597,11444,1.5452,3.47,7.60,2.19,67327.0
4,약정만료전 2~3개월,201385,3722,1.8482,0.94,2.47,2.62,18308.0
5,약정만료전 1~2개월,178577,4005,2.2427,0.84,2.66,3.17,16234.0
6,약정만료전 1개월,152700,3474,2.2750,0.72,2.31,3.22,13882.0
7,약정만료 1개월,302492,3746,1.2384,1.42,2.49,1.75,27499.0
8,약정만료후 1개월~2개월,117385,1130,0.9626,0.55,0.75,1.36,10671.0
9,약정만료후 2개월~3개월,109438,1208,1.1038,0.51,0.80,1.56,9949.0


##### Sizing the Pre-Expiry Window
- Combine the 1–6 months before expiry segments to size the target group and its efficiency

In [2]:
TARGET_WINDOWS = {
    "만료 1~3개월 전": ["약정만료전 1개월", "약정만료전 1~2개월", "약정만료전 2~3개월"],
    "만료 1~6개월 전": ["약정만료전 1개월", "약정만료전 1~2개월", "약정만료전 2~3개월", "약정만료전 3~6개월"],
}

total_n = expiry_share['n'].sum()
total_churn = expiry_share['n_churn'].sum()

rows = []
for name, segs in TARGET_WINDOWS.items():
    sub = expiry_share[expiry_share['AGMT_END_SEG'].isin(segs)]
    n, churn = sub['n'].sum(), sub['n_churn'].sum()
    rows.append({
        "window": name,
        "base_share_pct": round(n * 100 / total_n, 2),            # share of customer-months targeted
        "churn_share_pct": round(churn * 100 / total_churn, 2),    # share of all cancellations covered
        "concentration": round((churn / total_churn) / (n / total_n), 2),
        "monthly_churn_rate_pct": round(churn * 100 / n, 2),
        "customers_per_month": round(n / n_months),
        "cancellations_per_month": round(churn / n_months),
    })

target_windows = pd.DataFrame(rows)
target_windows

,window,base_share_pct,churn_share_pct,concentration,monthly_churn_rate_pct,customers_per_month,cancellations_per_month
0,만료 1~3개월 전,2.50,7.44,2.98,2.10,48424,1018
1,만료 1~6개월 전,5.97,15.04,2.52,1.78,115751,2059


##### Findings & Proposal
| Contract stage | Share of customers | Share of cancellations | Action |
|---|---|---|---|
| 1–6 months before expiry | 6% | 15% | Retention offer from 6 months before expiry (2.5x more efficient than random) |
| 12+ months after expiry | 35% | 36% | Out of contract but still subscribed → win back with re-contract offers |
| 12+ months before expiry | 43% | 28% | No contract trigger → early warning based on high-risk segments (Part 4) |

- The remaining ~21% falls between these stages (6–12 months before expiry, the expiry month, 1–12 months after expiry)
- Caveat: "covered" cancellations are an upper bound; the real effect needs an experiment to measure

---

## Part 4. Overlap Check: Compare Segments at the Same Contract Stage

##### Method
- Group customers with the same contract-expiry timing, then compare Top 5 segments' monthly churn within each group
- If a segment is still higher under the same contract timing, its risk is independent of the contract

In [3]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
CLEAN = "read_parquet('df_clean_part*.parquet')"

duckdb.sql(f"""
    CREATE OR REPLACE VIEW churn_base AS
    WITH first_cancel AS (
        SELECT sha2_hash, min(p_mt) AS first_cancel_pmt
        FROM {CLEAN}
        WHERE cancel_yn = '해지'
        GROUP BY sha2_hash
    )
    SELECT c.*
    FROM {CLEAN} c
    LEFT JOIN first_cancel fc USING (sha2_hash)
    WHERE fc.first_cancel_pmt IS NULL
       OR c.p_mt <= fc.first_cancel_pmt
""")

EXPIRY_GROUP = """
    CASE
        WHEN AGMT_END_SEG IN ('약정만료전 1개월', '약정만료전 1~2개월', '약정만료전 2~3개월', '약정만료전 3~6개월')
            THEN '1) 만료 1~6개월 전'
        WHEN AGMT_END_SEG IN ('약정만료전 6~9개월', '약정만료전 9~12개월') THEN '2) 만료 6~12개월 전'
        WHEN AGMT_END_SEG = '약정만료전 12개월이상' THEN '3) 만료 12개월 이상 전'
        ELSE '4) 만료 이후'
    END"""

SEGMENTS = {
    "연령대": "CASE WHEN AGE_GRP10 = '20대' THEN '20대' WHEN AGE_GRP10 = '30대' THEN '30대' ELSE '그 외' END",
    "가입 기간": """CASE WHEN SVC_USE_DAYS_GRP = '24개월~ 36개월미만' THEN '24~36개월'
                         WHEN SVC_USE_DAYS_GRP = '6개월미만' THEN '6개월 미만' ELSE '그 외' END""",
    "유료 채널 가입 수": """CASE WHEN PAID_CHNL_CNT_GRP IN ('2건', '3건 이상') THEN '2건 이상'
                                WHEN PAID_CHNL_CNT_GRP = '1건' THEN '1건' ELSE '0건' END""",
    "최근 시청일": "CASE WHEN CH_LAST_DAYS_BF_GRP = '3개월내없음' THEN '3개월 내 시청 없음' ELSE '그 외' END",
}


def within_expiry(segment_sql):
    long = duckdb.sql(f"""
        SELECT
            {EXPIRY_GROUP} AS expiry_group,
            {segment_sql} AS segment,
            count(*) AS n,
            round(avg(CASE WHEN cancel_yn = '해지' THEN 1 ELSE 0 END) * 100, 3) AS churn_rate_pct
        FROM churn_base
        GROUP BY 1, 2
    """).df()
    rates = long.pivot(index='segment', columns='expiry_group', values='churn_rate_pct')

    near = duckdb.sql(f"""
        SELECT
            {segment_sql} AS segment,
            round(avg(CASE WHEN {EXPIRY_GROUP} = '1) 만료 1~6개월 전' THEN 1 ELSE 0 END) * 100, 2)
                AS pct_near_expiry
        FROM churn_base
        GROUP BY 1
    """).df().set_index('segment')

    return rates.join(near)


overlap_results = {name: within_expiry(sql) for name, sql in SEGMENTS.items()}

for name, table in overlap_results.items():
    print(f"\n[{name}]  monthly churn rate (%) by expiry group, + share sitting 1~6 months before expiry")
    display(table)


[연령대]  monthly churn rate (%) by expiry group, + share sitting 1~6 months before expiry


,1) 만료 1~6개월 전,2) 만료 6~12개월 전,3) 만료 12개월 이상 전,4) 만료 이후,pct_near_expiry
segment,,,,,
20대,4.517,2.383,2.036,2.014,8.16
30대,3.715,1.890,0.918,1.424,7.98
그 외,1.630,0.779,0.415,0.754,5.87



[가입 기간]  monthly churn rate (%) by expiry group, + share sitting 1~6 months before expiry


,1) 만료 1~6개월 전,2) 만료 6~12개월 전,3) 만료 12개월 이상 전,4) 만료 이후,pct_near_expiry
segment,,,,,
24~36개월,2.536,1.304,0.352,1.332,31.32
6개월 미만,1.772,1.243,1.206,1.162,1.43
그 외,0.954,0.485,0.364,0.759,3.33



[유료 채널 가입 수]  monthly churn rate (%) by expiry group, + share sitting 1~6 months before expiry


,1) 만료 1~6개월 전,2) 만료 6~12개월 전,3) 만료 12개월 이상 전,4) 만료 이후,pct_near_expiry
segment,,,,,
0건,1.746,0.831,0.446,0.758,5.97
1건,2.682,1.223,0.770,1.281,5.97
2건 이상,2.369,1.590,1.310,1.408,5.87



[최근 시청일]  monthly churn rate (%) by expiry group, + share sitting 1~6 months before expiry


,1) 만료 1~6개월 전,2) 만료 6~12개월 전,3) 만료 12개월 이상 전,4) 만료 이후,pct_near_expiry
segment,,,,,
3개월 내 시청 없음,3.299,1.389,0.780,1.385,6.41
그 외,1.467,0.746,0.409,0.642,5.89


##### Findings
- **Risk independent of contract timing**: age 20s, paid-channel users, no viewing in 3 months, tenure under 6 months → higher churn than others in every contract group
- **Same signal as contract expiry**: tenure 24–36 months → 31% sit right before expiry, and with 12+ months to expiry there is no difference (0.35% vs 0.36%)
- Use: early-warning criteria for in-contract customers (28% of cancellations)
- Caveats: age 20s is only 0.8% of customers; only contract timing is controlled